# NLP Final: TF-IDF lost-item candidate ranking

This notebook demonstrates a reproducible **fictional** Strathmore-themed exercise. A lost report is compared with a pool of found reports and the system returns candidates for staff review. A score does not verify ownership or authorize release. The 2026 office inventories supply aggregate category counts only; none of the individual records below came from those inventories.

## Data-mining workflow

1. **Define the task:** retrieve and rank potentially related found reports for a lost report; staff make every ownership and release decision.
2. **Understand the sources:** record provenance and distinguish aggregate inventory counts from individual reports.
3. **Prepare and inspect:** generate the controlled fixture, validate required columns and labels, inspect missing text, duplicate IDs, category counts and description lengths.
4. **Build features and models:** compare a Logistic Regression pair baseline, TF-IDF cosine retrieval, structured attributes, keyword overlap and a validation-tuned hybrid.
5. **Validate and test:** split by item group, select settings on validation only, and report held-out ranking accuracy, Recall@5, MRR and no-match false suggestions.
6. **Demonstrate the system:** inspect explanations, try the optional local HTTP request, and use the companion Android app for the end-user workflow.

The evaluation data is authored fiction, so these steps demonstrate a repeatable workflow rather than measured campus effectiveness.

## 1. Sources and provenance

The team transcribed two Office category tables. Their quantities total 388 for June-July and 299 for August. The linked original Google Docs were inaccessible to the connected account, so the transcription still needs verification against the source. These aggregate counts are not individual item reports. The controlled lost and found descriptions below are fictional development examples; the team must review each example and label before submission. No verified same-item campus pairs are available in this repository.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv, json
ROOT = Path.cwd()
assert (ROOT / 'scripts/prepare_ranking_benchmark.py').exists(), 'Run from repository root'
from scripts.prepare_ranking_benchmark import main as prepare
from scripts.validate_ranking_dataset import validate
_NOTEBOOK_TMP = TemporaryDirectory(prefix='nlp-ranking-')
RUN_DIR = Path(_NOTEBOOK_TMP.name)
DATA = RUN_DIR / 'ranking'
prepare(DATA)
audit = validate(DATA)
assert audit['valid'], audit['errors']
print(json.dumps(audit, indent=2))

{
  "inventory_quantities": {
    "2026-06_07": 388,
    "2026-08": 299
  },
  "found_records": 24,
  "lost_reports": 27,
  "controlled_positive_labels": 24,
  "controlled_negative_labels": 168,
  "no_match_queries": 3,
  "candidate_pool_size": 24,
  "source_type": "AI-authored fictional controlled examples",
  "review": "Independent student review pending; no genuine same-item labels or campus performance."
}
{
  "counts": {
    "found_reports.csv": 24,
    "lost_reports.csv": 27,
    "match_labels.csv": 192,
    "evaluation_queries.csv": 27
  },
  "label_counts": {
    "1": 24,
    "0": 168
  },
  "errors": [],
  "valid": true,
  "note": "Structural checks do not verify physical identity, independent authorship, or label truth."
}


## 2. Data understanding, preparation and leakage control

The preparation script creates the four controlled tables from versioned fixtures. Structural validation checks required columns, IDs, pair references, split assignments and labels; it cannot prove that the invented reports describe real items. Each controlled item has one found record and a separately worded lost claim. Labels use the controlled item-group ID. Training, validation and test groups are disjoint; no pair is randomly split. No-match queries have no relevant found ID. The fixed candidate pool contains all 24 fictional found records.

In [2]:
def read_csv(name):
    with (DATA / name).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))
found = read_csv('found_reports.csv'); lost = read_csv('lost_reports.csv')
labels = read_csv('match_labels.csv'); queries = read_csv('evaluation_queries.csv')
for split in ('train', 'validation', 'test'):
    groups = {r['item_group_id'] for r in found if r['split'] == split}
    print(split, 'item groups:', len(groups), 'matched queries:', sum(q['split']==split and q['has_match']=='1' for q in queries))
print('sources:', sorted({r['source_type'] for r in found}))
print('example:', {k: found[0][k] for k in ('found_id','category','description','source_type')})

train item groups: 8 matched queries: 8
validation item groups: 8 matched queries: 8
test item groups: 8 matched queries: 8
sources: ['ai_constructed_fictional']
example: {'found_id': 'f-g001', 'category': 'plastic_water_bottle', 'description': 'Clear plastic bottle with orange flip lid', 'source_type': 'ai_constructed_fictional'}


In [3]:
from collections import Counter
from statistics import median

def inspect_table(name, rows, id_field):
    ids = [row.get(id_field, '').strip() for row in rows]
    descriptions = [row.get('description', '').strip() for row in rows]
    categories = Counter(row.get('category', '').strip() or '[missing]' for row in rows)
    print(f'{name}: {len(rows)} rows; duplicate {id_field}: {len(ids) - len(set(ids))}; blank descriptions: {sum(not text for text in descriptions)}')
    print(f'  description characters: min={min(map(len, descriptions))}, median={median(map(len, descriptions))}, max={max(map(len, descriptions))}')
    print('  category counts:', dict(sorted(categories.items())))

inspect_table('found', found, 'found_id')
inspect_table('lost', lost, 'lost_id')
print('pair-label counts:', dict(Counter(row['label'] for row in labels)))
print('query split counts:', dict(Counter(row['split'] for row in queries)))

found: 24 rows; duplicate found_id: 0; blank descriptions: 0
  description characters: min=34, median=42.5, max=54
  category counts: {'calculator': 2, 'headphones': 1, 'laptop_charger': 3, 'metallic_bottle': 3, 'mouse': 2, 'pencil_pouch': 3, 'plastic_water_bottle': 3, 'spectacle_case': 1, 'spectacles': 1, 'sweater': 1, 'umbrella': 1, 'usb_cable': 3}
lost: 27 rows; duplicate lost_id: 0; blank descriptions: 0
  description characters: min=28, median=54, max=67
  category counts: {'calculator': 2, 'headphones': 1, 'laptop_charger': 3, 'metallic_bottle': 3, 'mouse': 2, 'pencil_pouch': 3, 'plastic_water_bottle': 3, 'scarf': 1, 'spectacle_case': 1, 'spectacles': 1, 'sweater': 1, 'umbrella': 2, 'usb_cable': 3, 'watch': 1}
pair-label counts: {'1': 24, '0': 168}
query split counts: {'train': 8, 'validation': 9, 'test': 10}


## 3. Logistic Regression comparison

The earlier CAT 2 coursework used Logistic Regression on engineered text and attribute comparisons. This benchmark fits a fresh Logistic Regression model on its 64 training pairs; it does not load the older notebook's model or artifact. Its scores rank the same candidate pool as the retrieval methods and remain results on constructed labels.

In [4]:
from matching_system.logistic_regression_features import FEATURE_NAMES, pair_features
from scripts.evaluate_ranking_benchmark import as_report
train_labels = [r for r in labels if r['split']=='train']
found_by_id = {r['found_id']: r for r in found}; lost_by_id = {r['lost_id']: r for r in lost}
example = train_labels[0]
features = pair_features(as_report(lost_by_id[example['lost_id']], 'lost'), as_report(found_by_id[example['found_id']], 'found'))
print('training pairs:', len(train_labels), 'features:', list(features))

training pairs: 64 features: ['description_similarity', 'category_match', 'category_conflict', 'colour_match', 'colour_conflict', 'brand_match', 'brand_conflict', 'model_match', 'model_conflict', 'material_match', 'material_conflict', 'distinctive_feature_similarity', 'location_similarity', 'time_difference_hours']


## 4. NLP text features and cosine similarity

TF-IDF gives more weight to terms that distinguish documents in the fitted corpus. Unigrams and bigrams retain individual words and short phrases. The vocabulary below is fitted only on training descriptions and then reused to transform validation, test and candidate records. Cosine similarity measures the angle between their feature vectors; it is textual relevance, not a probability of ownership.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from matching_system.nlp_retrieval import normalize_report_text
training_text = [normalize_report_text(r['description']) for r in found + lost if r['split']=='train']
vectorizer = TfidfVectorizer(ngram_range=(1,2), strip_accents='unicode').fit(training_text)
query_text = normalize_report_text(lost_by_id['l-g017']['description'])
found_text = normalize_report_text(found_by_id['f-g017']['description'])
similarity = cosine_similarity(vectorizer.transform([query_text]), vectorizer.transform([found_text]))[0,0]
print('vocabulary size:', len(vectorizer.vocabulary_), 'example cosine:', round(float(similarity), 4))

vocabulary size: 205 example cosine: 0.1682


## 5. Hybrid ranking and validation

The comparison script applies one candidate pool to keyword overlap, attribute scoring, Logistic Regression, TF-IDF cosine and a weighted hybrid. It selects a hybrid weight from 0.25, 0.50 or 0.75 using validation Recall@5 and MRR. A strong-suggestion cutoff is also derived from validation no-match scores. The test split is used only for reporting.

In [6]:
from scripts.evaluate_ranking_benchmark import main as evaluate
results = evaluate(data_dir=DATA, result_path=RUN_DIR/'ranking_benchmark.json')
print('selected TF-IDF weight:', results['hybrid']['tfidf_weight_selected_on_validation'])
print('training feature count:', len(results['logistic_regression']['features']))

{
  "keyword": {
    "matched_queries": 8,
    "no_match_queries": 2,
    "candidate_pool_size": 24,
    "top1_accuracy": 0.25,
    "recall_at_5": 0.875,
    "mrr": 0.540625,
    "matched_details": [
      {
        "query_id": "q-g017",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g018",
        "first_rank": 8,
        "recall_at_5": 0.0
      },
      {
        "query_id": "q-g019",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g020",
        "first_rank": 5,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g021",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g022",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g023",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g024",
        "first_rank": 2,
        "recall_at_5": 1.0
      }
    ],
  

## 6. Held-out ranking results

Top-1 accuracy is the fraction of matched held-out queries whose relevant found record ranks first. Recall@5 is the fraction with a relevant record in the first five. Mean reciprocal rank (MRR) averages the reciprocal rank of the first relevant record. The final column counts no-match queries whose top score exceeds a cutoff selected on validation data. Matched-query denominators and no-match denominators are shown separately. These are controlled-fiction results, not campus recovery rates.

In [7]:
print('method | Top-1 accuracy (matched queries) | Recall@5 | MRR | no-match false suggestions')
for name, item in results['methods'].items():
    test = item['test']
    print(f"{name} | {test['top1_accuracy']:.3f} ({test['matched_queries']}) | {test['recall_at_5']:.3f} | {test['mrr']:.3f} | {item['test_no_match_false_suggestions']}/{item['test_no_match_denominator']}")

method | Top-1 accuracy (matched queries) | Recall@5 | MRR | no-match false suggestions
keyword | 0.250 (8) | 0.875 | 0.541 | 1/2
attributes | 1.000 (8) | 1.000 | 1.000 | 1/2
logistic_regression | 1.000 (8) | 1.000 | 1.000 | 1/2
tfidf | 0.250 (8) | 0.500 | 0.391 | 2/2
hybrid | 1.000 (8) | 1.000 | 1.000 | 2/2


## 7. Explainability and offline application smoke check

The runtime `MatchingPipeline` returns lexical similarity alongside structured fact statuses and staff-workflow guidance. This cell calls the FastAPI preview function directly with an invented request. It checks model-to-explanation behavior but does not test HTTP routing, the authenticated Firebase service, or Android connectivity. A score is text evidence, not a probability or release decision.

In [8]:
from pydantic import ValidationError
from matching_system.api import PreviewRequest, preview_matches
request = PreviewRequest(item_name='Metal water bottle',description='Blue bottle with a scratched lid',category_id='metal_water_bottle',location_id='msb',event_date='2026-11-10',limit=3)
body = preview_matches(request)
print('status:', body['status'], 'first ID:', body['matches'][0]['found_report_id'])
print('explanation:', body['matches'][0]['explanation'])
for label, values in [('blank description', {'item_name':'Water bottle','description':''}), ('invalid calendar date', {'item_name':'Water bottle','description':'Blue bottle','event_date':'2026-02-30'})]:
    try:
        PreviewRequest(**values)
    except ValidationError:
        print(label + ': rejected')

status: possible_matches first ID: demo-found-metal
explanation: Facts: category agrees; location differs; event date agrees. Staff should review the conflicting details. Text similarity is lexical evidence, not a probability. Ownership is unverified; release is not authorized.
blank description: rejected
invalid calendar date: rejected


## 8. Optional live HTTP preview

For an end-user-like request through the running local FastAPI server, start this in a separate terminal from the repository root: `python -m uvicorn matching_system.api:app --host 127.0.0.1 --port 8000`. Set `NLP_LIVE_HTTP_DEMO=1` in the notebook process, then run the cell below. It submits only fictional general-item text to the local, non-persistent preview API and checks valid, blank-text and invalid-date responses. Leave the flag unset to keep a normal offline notebook run self-contained. This preview is not the authenticated mobile workflow; the companion Android guide explains how to run the app through Firebase emulators and the worker.

In [9]:
import os
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def post_preview(payload):
    req = Request('http://127.0.0.1:8000/preview/matches', data=json.dumps(payload).encode('utf-8'), headers={'Content-Type':'application/json'}, method='POST')
    try:
        with urlopen(req, timeout=5) as response:
            return response.status, json.loads(response.read().decode('utf-8'))
    except HTTPError as error:
        return error.code, json.loads(error.read().decode('utf-8'))

if os.environ.get('NLP_LIVE_HTTP_DEMO') == '1':
    valid = {'item_name':'Metal water bottle','description':'Blue bottle with a scratched lid','category_id':'metal_water_bottle','location_id':'msb','event_date':'2026-11-10','limit':3}
    status, result = post_preview(valid)
    print('valid request:', status, result['status'], 'candidate count:', len(result['matches']))
    print('first explanation:', result['matches'][0]['explanation'] if result['matches'] else 'No candidate in the demo inventory')
    print('blank description:', post_preview({**valid, 'description':''})[0])
    print('invalid date:', post_preview({**valid, 'event_date':'2026-02-30'})[0])
else:
    print('Live HTTP check skipped. Start the local API and set NLP_LIVE_HTTP_DEMO=1 to run it.')

valid request: 200 possible_matches candidate count: 3
first explanation: Facts: category agrees; location differs; event date agrees. Staff should review the conflicting details. Text similarity is lexical evidence, not a probability. Ownership is unverified; release is not authorized.
blank description: 422
invalid date: 422


## 9. Interpretation, results and limitations

The structured and hybrid methods score perfectly on the eight matched fictional test queries because the generated lost and found records share category and colour fields by construction. That is a benchmark design weakness, not evidence of field effectiveness. TF-IDF alone misses several paraphrases after fitting on only eight training groups. The held-out TF-IDF result also raises a no-match risk: its top candidate exceeded the validation cutoff on both test no-match queries. That is 2/2 for this tiny fictional sample and must not be presented as a real-world false-positive rate.

A new independently collected, permitted and independently verified set of item identities, with a frozen holdout, is required before claiming campus performance. The team must verify the aggregate source transcription, review the fictional examples and labels, run the live app walkthrough, and approve the notebook for submission.